Install dependencies from uv.

In [1]:
!uv sync

Resolved 103 packages in 10ms
Checked 100 packages in 203ms


Load environment variables from .env file.

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

Create the weather tool.

In [3]:
import requests

def get_weather(latitude: float, longitude: float) -> dict:
    """
    Fetch weather forecast from the National Weather Service API.
    
    Args:
        latitude: The latitude of the location
        longitude: The longitude of the location
        
    Returns:
        A dictionary containing the weather forecast data
        
    Raises:
        requests.HTTPError: If the API request fails
        ValueError: If the coordinates are invalid
    """
    if not (-90 <= latitude <= 90) or not (-180 <= longitude <= 180):
        raise ValueError("Invalid coordinates. Latitude must be between -90 and 90, "
                         "longitude must be between -180 and 180.")

    headers = {
        "User-Agent": "WeatherApp/1.0 (your@email.com)",
        "Accept": "application/geo+json"
    }

    # Step 1: Get the grid points for the given coordinates
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    response = requests.get(points_url, headers=headers)
    response.raise_for_status()
    points_data = response.json()

    properties = points_data.get("properties", {})
    forecast_url = properties.get("forecast")
    location_info = {
        "city": properties.get("relativeLocation", {}).get("properties", {}).get("city"),
        "state": properties.get("relativeLocation", {}).get("properties", {}).get("state"),
        "grid_office": properties.get("gridId"),
    }

    if not forecast_url:
        raise ValueError("Could not retrieve forecast URL from NWS API.")

    # Step 2: Get the forecast using the forecast URL
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    periods = forecast_data.get("properties", {}).get("periods", [])

    return {
        "location": location_info,
        "forecast": [
            {
                "name": period.get("name"),
                "temperature": period.get("temperature"),
                "temperature_unit": period.get("temperatureUnit"),
                "wind_speed": period.get("windSpeed"),
                "wind_direction": period.get("windDirection"),
                "short_forecast": period.get("shortForecast"),
                "detailed_forecast": period.get("detailedForecast"),
                "is_daytime": period.get("isDaytime"),
            }
            for period in periods
        ],
    }

Get coordinates for New York City.

In [4]:
get_weather(40.7143, -74.006)

{'location': {'city': 'New York', 'state': 'NY', 'grid_office': 'OKX'},
 'forecast': [{'name': 'This Afternoon',
   'temperature': 87,
   'temperature_unit': 'F',
   'wind_speed': '12 mph',
   'wind_direction': 'SW',
   'short_forecast': 'Slight Chance Showers And Thunderstorms',
   'detailed_forecast': 'A slight chance of showers and thunderstorms before 4pm. Mostly sunny. High near 87, with temperatures falling to around 85 in the afternoon. Heat index values as high as 97. Southwest wind around 12 mph. Chance of precipitation is 20%. New rainfall amounts less than a tenth of an inch possible.',
   'is_daytime': True},
  {'name': 'Tonight',
   'temperature': 77,
   'temperature_unit': 'F',
   'wind_speed': '6 to 10 mph',
   'wind_direction': 'SW',
   'short_forecast': 'Slight Chance Showers And Thunderstorms',
   'detailed_forecast': 'A slight chance of showers and thunderstorms before 2am. Partly cloudy, with a low around 77. Southwest wind 6 to 10 mph. Chance of precipitation is 20

Create tool for getting lattidue and longitude from google.

In [5]:
import urllib.request
import json
import os

GOOGLE_MAPS_KEY = os.getenv("GOOGLE_MAPS_KEY", "")

if not GOOGLE_MAPS_KEY:
    raise Exception("Google maps key must be set in environment.")

def get_lat_lon(city: str, state: str) -> tuple[float, float]:
    """
    Fetch latitude and longitude for a given city and state using Google Geocoding API.

    Args:
        city: City name (e.g. "Austin")
        state: State name or abbreviation (e.g. "TX" or "Texas")

    Returns:
        A tuple of (latitude, longitude)

    Raises:
        ValueError: If the location is not found or the API returns an error
    """
    address = urllib.parse.quote(f"{city}, {state}")
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={GOOGLE_MAPS_KEY}"

    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode())

    if data["status"] != "OK":
        raise ValueError(f"Geocoding API error: {data['status']} for '{city}, {state}'")

    location = data["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]

Test getting lattidue and longitude.

In [6]:
get_lat_lon("New York City", "New York")

(40.7127753, -74.0059728)

Create the agents.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search, agent_tool

weather_agent_instructions = """
You are an assistent to help with getting the weather. You will start by asking the user what city and state they want the weather for.
Use this to call the get_lat_long tool and get the latitude and longitude. Use this to call the get_weather tool to get the weather response.
Convert this response into human readable text.
"""

weather_agent = Agent(
    name="weather_agent",
    model="gemini-flash-latest",
    instruction=weather_agent_instructions,
    tools=[get_weather, get_lat_lon]
)

search_agent_instructions = """
# Search Agent System Prompt

You are an intelligent search agent designed to find, retrieve, and synthesize information efficiently and accurately. Your primary goal is to help users get precise, relevant answers by conducting thorough searches and presenting results clearly.

## Core Responsibilities

- **Query Analysis**: Break down complex user queries into effective search terms
- **Information Retrieval**: Search across available sources to find the most relevant results
- **Synthesis**: Combine information from multiple sources into coherent, accurate responses
- **Citation**: Always attribute information to its source

## Behavior Guidelines

### When Searching
- Reformulate vague queries into specific, targeted search terms
- Perform multiple searches if the first query yields insufficient results
- Broaden or narrow search scope based on the quality of initial results
- Prioritize authoritative, recent, and relevant sources

### When Responding
- Lead with the most relevant answer before elaborating
- Clearly distinguish between established facts and uncertain information
- Use phrases like *"According to [source]..."* or *"Based on available results..."*
- Flag outdated or potentially unreliable information
- Summarize lengthy results into digestible key points

### When to Clarify
Ask for clarification when:
- The query is ambiguous or has multiple interpretations
- The topic requires a specific time range or region
- The user's intent is unclear (e.g., research vs. quick fact-check)

## Constraints & Limitations

- Do **not** fabricate sources, URLs, or citations
- Do **not** present search results as personal opinions
- Do **not** access restricted, private, or paywalled content
- Always acknowledge when a topic falls outside your search capabilities
- Respect content boundaries — avoid retrieving harmful or illegal content

## Output Format

Structure your responses as follows:

1. **Direct Answer** — A concise response to the query (1–3 sentences)
2. **Supporting Details** — Expanded context, evidence, or explanation
3. **Sources** — List of references used (title, URL, date if available)
4. **Follow-up Suggestions** *(optional)* — Related searches the user might find helpful

## Tone & Style

- Professional, neutral, and objective
- Concise but thorough — avoid unnecessary filler
- Adapt complexity to the user's apparent level of expertise
- Use bullet points and headers for multi-part answers
"""

search_agent = Agent(name="search_agent",
                     model="gemini-flash-latest",
                     instruction=search_agent_instructions,
                     tools=[google_search])

root_agent_instructions = """
# Router Agent System Prompt

You are an intelligent routing agent responsible for analyzing incoming user requests and directing them to the most appropriate specialized agent. You currently have access to two agents: a **Weather Agent** and a **Search Agent**. Your job is to ensure every request reaches the right agent quickly and accurately.

## Available Agents

### 🌤️ Weather Agent
Handles all weather-related requests including:
- Current weather conditions for a location
- Weather forecasts (hourly, daily, weekly)
- Severe weather alerts and warnings
- Historical weather data
- Climate and seasonal information
- Weather-related travel advisories

### 🔍 Search Agent
Handles all general information retrieval requests including:
- Factual questions and knowledge lookups
- News and current events
- Product, service, or business information
- Research and academic topics
- How-to guides and instructions
- People, places, organizations, and events

---

## Routing Rules

### Route to Weather Agent when the request:
- Mentions weather, temperature, forecast, humidity, wind, precipitation, or storm
- Asks "Will it rain?", "Is it cold in...?", "What's the weather like in...?"
- References weather phenomena (hurricane, tornado, snow, fog, heatwave, etc.)
- Asks about the best time to visit a location based on climate

### Route to Search Agent when the request:
- Asks for facts, definitions, or explanations
- Requests news, articles, or web-based information
- Involves researching a topic, product, person, or event
- Cannot be answered purely with weather data

### When a Request Spans Both Agents
Some requests may require both agents. For example:
- *"What's the weather in Paris and what are the top attractions?"*
  → Route to **Weather Agent** for weather, then **Search Agent** for attractions
- *"Is it a good weekend to hike Mount Fuji?"*
  → Route to **Weather Agent** for conditions, **Search Agent** for trail/hiking info

In these cases, fan out to both agents in parallel and synthesize the results into a single unified response.

---

## Routing Behavior

1. **Analyze** the user's request to identify the core intent
2. **Identify** which agent(s) are best suited to handle it
3. **Extract** any key parameters needed by the target agent (e.g., location, date, topic)
4. **Dispatch** the request with a well-formed query to the appropriate agent
5. **Return** the agent's response to the user — do not alter or fabricate the content

---

## Handling Edge Cases

| Scenario | Action |
|---|---|
| Request is unclear or ambiguous | Ask the user a single clarifying question before routing |
| Request doesn't match any agent | Inform the user this falls outside available capabilities |
| An agent returns an error or no results | Notify the user and suggest rephrasing the request |
| Request is harmful or inappropriate | Decline politely and do not route |

---

## Constraints

- Do **not** answer questions directly — your role is to route, not respond
- Do **not** modify the user's core request when passing it to an agent
- Do **not** combine or fabricate information beyond what agents return
- Always maintain a **neutral, transparent** tone when explaining routing decisions

---

## Output Format

For every routed request, follow this internal structure:

```json
{
  "intent": "<brief description of user intent>",
  "route_to": ["weather_agent" | "search_agent" | "both"],
  "parameters": {
    "location": "<if applicable>",
    "date": "<if applicable>",
    "query": "<reformulated query for the target agent>"
  }
}
```

The final response returned to the user should be clean and natural — do not expose the routing JSON unless asked.

---

## Example Routing Decisions

| User Request | Route To |
|---|---|
| "What's the weather in Tokyo tomorrow?" | 🌤️ Weather Agent |
| "Who invented the telephone?" | 🔍 Search Agent |
| "Is it going to snow in Denver this week?" | 🌤️ Weather Agent |
| "What are the best restaurants in Austin?" | 🔍 Search Agent |
| "Should I bring an umbrella in London and what museums are nearby?" | 🌤️ + 🔍 Both |
| "What's the climate like in Bali in July and what should I pack?" | 🌤️ + 🔍 Both |
"""

root_agent = Agent(
    name="root_agent",
    model="gemini-flash-latest",
    instruction=root_agent_instructions,
    tools=[agent_tool.AgentTool(agent=search_agent)],
    sub_agents=[weather_agent]
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (117477511.py, line 198)

Setup the runner.

In [15]:
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

session_service = InMemorySessionService()

runner = Runner(
    agent=root_agent,
    app_name="root_app",
    session_service=session_service,
)


async def run_prompt(prompt: str):
    session = await session_service.create_session(
        app_name="root_app",
        user_id="user_123",
    )

    content = types.Content(
        role="user",
        parts=[types.Part(text=prompt)])

    async for event in runner.run_async(user_id="user_123",
                                        session_id=session.id,
                                        new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                print(event.content.parts[0].text)

Tests for getting weather.

In [16]:
print("================ New York ===========================")
await run_prompt("What is the weather for New York City, New York")
print("================ Reston =============================")
await run_prompt("What is the weather for Reston, VA")
print("================ Los Angeles ========================")
await run_prompt("What is the weather for Los Angeles, CA")

================ New York ===========================
Here is the current weather forecast for New York City, New York:

* **This Afternoon:** Mostly sunny with a high near 87°F (heat index up to 97°F). There is a 20% chance of showers and thunderstorms before 4 PM. Southwest winds around 12 mph.
* **Tonight:** Partly cloudy with a low around 77°F. Slight (20%) chance of showers and thunderstorms before 2 AM. Southwest winds around 6 to 10 mph.
* **Friday:** Mostly sunny with a high near 91°F (heat index up to 103°F). A 40% chance of showers and thunderstorms after 2 PM. Southwest winds 5 to 9 mph.
* **Friday Night:** Mostly cloudy with a low around 76°F and a 50% chance of showers and thunderstorms before 2 AM.
* **Weekend Outlook:** Highs around 91°F each day with mostly sunny conditions. A slight chance of afternoon thunderstorms on Saturday, followed by clear, sunny weather on Sunday.
================ Reston =============================
Here is the weather forecast for **Reston, V

Perform tests for searching.

In [17]:
print("================ Time ===========================")
await run_prompt("What is the current time in Tokyo")

================ Time ===========================


Tools at indices [0] are not compatible with automatic function calling (AFC). AFC is disabled. If AFC is intended, please include python callables in the tool list, and do not include function declaration and MCP server in the tool list.
Node execution failed with exception
Traceback (most recent call last):
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/Users/carpenterju/work/google-training/adk-workshop-jc/.venv/lib/python3.13/site-packages/google/adk/workflow/_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
      self._track_event_in_context(event, ctx)
      await self._enqueue_event(event

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Please enable tool_config.include_server_side_tool_invocations to use Built-in tools with Function calling.', 'status': 'INVALID_ARGUMENT'}}